In [9]:
# First cell - Data preparation and risk classification generation
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load training and testing datasets
train_data = pd.read_csv("train_dataset.csv")
test_data = pd.read_csv("test_dataset.csv")

# Define risk classification rules
def generate_risk_classification(row):
    # Calculate risk score based on features (customize these weights based on domain knowledge)
    risk_score = (
        row['traffic_congestion_level'] * 0.2 +
        row['port_congestion_level'] * 0.2 +
        row['delay_probability'] * 0.2 +
        row['route_risk_level'] * 0.15 +
        row['supplier_reliability_score'] * 0.15 +
        row['disruption_likelihood_score'] * 0.1
    )
    
    # Define risk thresholds
    if risk_score >= 0.7:
        return 'High Risk'
    elif risk_score >= 0.4:
        return 'Moderate Risk'
    else:
        return 'Low Risk'

# Generate risk classifications
train_data['risk_classification'] = train_data.apply(generate_risk_classification, axis=1)
test_data['risk_classification'] = test_data.apply(generate_risk_classification, axis=1)

# Select relevant features
features = [
    "traffic_congestion_level", "port_congestion_level", "delay_probability", 
    "route_risk_level", "supplier_reliability_score", "disruption_likelihood_score"
]
target = "risk_classification"

# Encode categorical target variable
le = LabelEncoder()
train_data[target] = le.fit_transform(train_data[target])
test_data[target] = le.transform(test_data[target])

# Split dataset
X_train, y_train = train_data[features], train_data[target]
X_test, y_test = test_data[features], test_data[target]

# Directory to save model checkpoints
os.makedirs("model_checkpoints", exist_ok=True)

In [10]:
# Train Random Forest Model with checkpoint saving
def train_model(save_every=100):
    model = RandomForestClassifier(n_estimators=100, random_state=42, warm_start=True)
    
    for epoch in range(1, 1001):  # Training for 50 epochs
        model.n_estimators += 100  # Increment trees for each epoch
        model.fit(X_train, y_train)
        
        if epoch % save_every == 0:
            checkpoint_path = f"model_checkpoints/risk_model_epoch_{epoch}.pkl"
            joblib.dump(model, checkpoint_path)
            print(f"Checkpoint saved at epoch {epoch}: {checkpoint_path}")

    return model

# Train the model
model = train_model()

# Make predictions and evaluate
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Calculate log loss
def log_loss(y_true, y_pred_proba):
    n_samples = len(y_true)
    loss = 0
    for i in range(n_samples):
        true_class = y_true[i]
        loss -= np.log(y_pred_proba[i][true_class])
    return loss / n_samples

loss = log_loss(y_test, y_pred_proba)
print(f"Log Loss: {loss:.4f}")

# Print classification report
print("\nClassification Report:")
class_names = le.classes_
print(classification_report(y_test, y_pred, target_names=class_names))

# Save final model
joblib.dump(model, "final_risk_model.pkl")
print("Final model saved as final_risk_model.pkl")

Checkpoint saved at epoch 100: model_checkpoints/risk_model_epoch_100.pkl
Checkpoint saved at epoch 200: model_checkpoints/risk_model_epoch_200.pkl
Checkpoint saved at epoch 300: model_checkpoints/risk_model_epoch_300.pkl
Checkpoint saved at epoch 400: model_checkpoints/risk_model_epoch_400.pkl
Checkpoint saved at epoch 500: model_checkpoints/risk_model_epoch_500.pkl
Checkpoint saved at epoch 600: model_checkpoints/risk_model_epoch_600.pkl
Checkpoint saved at epoch 700: model_checkpoints/risk_model_epoch_700.pkl
Checkpoint saved at epoch 800: model_checkpoints/risk_model_epoch_800.pkl
Checkpoint saved at epoch 900: model_checkpoints/risk_model_epoch_900.pkl
Checkpoint saved at epoch 1000: model_checkpoints/risk_model_epoch_1000.pkl
Accuracy: 0.9972
Log Loss: 0.0060

Classification Report:
               precision    recall  f1-score   support

    High Risk       1.00      1.00      1.00      6387
     Low Risk       0.50      0.25      0.33         4
Moderate Risk       0.70      0.32

In [11]:
def test_model(model_path="final_risk_model.pkl"):
    model = joblib.load(model_path)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=le.classes_)
    
    print(f"Model Accuracy: {accuracy:.2f}")
    print("Classification Report:\n", report)

# Test the final model
test_model()

Model Accuracy: 1.00
Classification Report:
                precision    recall  f1-score   support

    High Risk       1.00      1.00      1.00      6387
     Low Risk       0.50      0.25      0.33         4
Moderate Risk       0.70      0.32      0.44        22

     accuracy                           1.00      6413
    macro avg       0.73      0.52      0.59      6413
 weighted avg       1.00      1.00      1.00      6413

